In [ ]:
import re
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Load Data
# -----------------------------
train_df = pd.read_json("train.json")
test_df = pd.read_json("test.json")

# -----------------------------
# Dedup BEFORE preprocessing
# -----------------------------
train_df = train_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])
test_df = test_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])

# -----------------------------
# Preprocessing (HTML only — BERT needs natural language)
# -----------------------------
def preprocess(text):
    text = re.sub(r"<.*?>", " ", text)   # remove HTML tags only
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text"] = train_df["text"].apply(preprocess)
test_df["text"] = test_df["text"].apply(preprocess)

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True, max_length=512)

# -----------------------------
# Convert to HuggingFace Dataset
# -----------------------------
train_dataset = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
test_dataset = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize, batched=True)

# -----------------------------
# Model
# -----------------------------
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# -----------------------------
# Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                        # faster if GPU supports it
    logging_dir="./logs",
    logging_steps=100,
)

# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# -----------------------------
# Save Model
# -----------------------------
model.save_pretrained("distilbert_imdb")
tokenizer.save_pretrained("distilbert_imdb")
print("Model saved as distilbert_imdb/")

# -----------------------------
# Predict on test dataset
# -----------------------------
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert_imdb",
    tokenizer="distilbert_imdb"
)

test_texts = test_df["text"].tolist()
raw_predictions = classifier(test_texts, truncation=True, max_length=512, batch_size=32)

test_predictions = [1 if p["label"] == "LABEL_1" else 0 for p in raw_predictions]
test_probabilities = [p["score"] for p in raw_predictions]

# -----------------------------
# Evaluation
# -----------------------------
y_test = test_df["label"].tolist()
print("Accuracy:", accuracy_score(y_test, test_predictions))
print("\nClassification Report:\n", classification_report(y_test, test_predictions))

# -----------------------------
# Show some predictions
# -----------------------------
for i in range(5):
    text = test_df["text"].iloc[i]
    pred_label = "Positive" if test_predictions[i] == 1 else "Negative"
    print("Review:", text[:200], "...")
    print("Prediction:", pred_label)
    print("Confidence Score:", test_probabilities[i])
    print("-" * 60)

Map:   0%|          | 0/24904 [00:00<?, ? examples/s]

Map:   0%|          | 0/24801 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/home/noob/miniconda3/envs/ai/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument i